# Bielik - dwa agenty z [LlamaIndex](https://www.llamaindex.ai/): ReAct i function calling

[Bielik](https://bielik.ai/) to polski model językowy stworzony przez [SpeakLeash](https://speakleash.org/) i ICM. W tym notebooku zbudujemy agenta pogodowego (analogicznego do tego z `003-01. LLM-function_calling.ipynb`) opartego o model `SpeakLeash/bielik-11b-v3.0-instruct:bf16` hostowany lokalnie z użyciem [Ollama](https://ollama.com/).

[LlamaIndex](https://www.llamaindex.ai/) to popularny framework do budowania aplikacji LLM (głównie kojarzony z RAG-iem, ale dostarcza też pełną warstwę agentową). W odróżnieniu od DSPy, LlamaIndex ma **kilka gotowych klas agentów** - tutaj pokażemy dwie najpopularniejsze:

- **Wariant 1 - `ReActAgent`** - agent z pętlą Thought/Action/Observation w tekście, parsowaną po stronie frameworka. Działa z każdym modelem.
- **Wariant 2 - `FunctionAgent`** - agent oparty o natywny function calling LLMa (strukturyzowane pole `tool_calls` w odpowiedzi). Wymaga, żeby model i serwer poprawnie zwracali `tool_calls` - co dla Bielika na Ollamie udało się dopiero po wgraniu customowego Modelfile (`010-00. Bielik.Modelfile`).

Pozostałe trzy notebooki w tej serii pokazują tę samą funkcjonalność w innych podejściach:
- `010-01. Bielik-openai.ipynb` - manualna pętla function callingu na surowym SDK
- `010-02. Bielik-pydantic-ai.ipynb` - function calling z PydanticAI
- `010-03. Bielik-dspy.ipynb` - ReAct z DSPy

Różnice względem DSPy (notebook `010-03`):
- LlamaIndex jest bliżej "imperatywnego" stylu - pętla agenta i narzędzia są jawnymi obiektami,
- DSPy traktuje opis zadania deklaratywnie poprzez sygnaturę i potrafi optymalizować promptu (np. teleprompter z `009-01`),
- LlamaIndex ma bogatszy ekosystem integracji z bazami wektorowymi, dokumentami i toolami zewnętrznymi - oraz więcej gotowych klas agentów (`ReActAgent`, `FunctionAgent`, `CodeActAgent`, `AgentWorkflow`).

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile:
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
   Modelfile naprawia bug ze stop tokens (oryginał używa tokenów Llamy 3 zamiast ChatML, co psuje generację) **i** włącza strukturyzowane parsowanie `tool_calls` po stronie Ollamy - bez niego `FunctionAgent` w wariancie 2 nie zadziała. Szczegóły w `010-00. Bielik.Modelfile.md`.
4. Działający serwis Ollama w tle (port `11434`).

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

In [ ]:
import requests
import json

## Funkcja pobierająca współrzędne geograficzne dla danej nazwy

In [ ]:
def get_geolocation(location):
    """
    Pobiera współrzędne geograficzne oraz dane lokalizacyjne.

    Parametry:
    location (str): Nazwa lokalizacji, dla której chcemy uzyskać współrzędne geograficzne.

    Zwraca:
    dict: Dane lokalizacyjne w formacie JSON.
    """
    print(f"[tool_call] get_geolocation(location={location!r})")

    geocode_endpoint = "https://nominatim.openstreetmap.org/search"
    geocode_params = {"q": location, "format": "json"}
    headers = {"User-Agent": "Python script"}

    geocode_response = requests.get(geocode_endpoint, params=geocode_params, headers=headers)
    geocode_data = geocode_response.json()

    simplified_data = {
        "name": geocode_data[0]["display_name"],
        "latitude": geocode_data[0]["lat"],
        "longitude": geocode_data[0]["lon"]
    }
    return simplified_data

geolocation_data = get_geolocation("Poznań")
print(json.dumps(geolocation_data, indent=4, ensure_ascii=False))

## Funkcja pobierająca informacje o pogodzie dla podanych współrzędnych geograficznych

In [ ]:
def get_wind_direction(degrees):
    """Konwertuje kierunek wiatru ze stopni na nazwy kierunków świata."""
    directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                  'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    index = int((degrees + 11.25) // 22.5) % 16
    return directions[index]

def get_current_weather(latitude, longitude):
    """
    Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych.

    Parametry:
    latitude (float): Szerokość geograficzna.
    longitude (float): Długość geograficzna.

    Zwraca:
    dict: Dane pogodowe w formacie JSON.
    """
    print(f"[tool_call] get_current_weather(latitude={latitude!r}, longitude={longitude!r})")

    weather_endpoint = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    weather_response = requests.get(weather_endpoint, params=weather_params)
    weather_data = weather_response.json()

    simplified_weather = {
        "temperature": f"{weather_data['current_weather']['temperature']} °C",
        "wind_speed": f"{weather_data['current_weather']['windspeed']} km/h",
        "wind_direction": get_wind_direction(weather_data['current_weather']['winddirection']),
        "is_day": "day" if weather_data['current_weather']['is_day'] else "night"
    }

    return simplified_weather

current_weather = get_current_weather(geolocation_data['latitude'], geolocation_data['longitude'])
print(json.dumps(current_weather, indent=4))

# Wariant 1: agent ReAct (`ReActAgent`)

Pętla Thought/Action/Observation parsowana po stronie frameworka. Działa nawet wtedy, gdy serwer LLM nie zwraca strukturyzowanych `tool_calls` (jak Bielik na Ollamie bez customowego Modelfile). Klasyczne, sprawdzone podejście do agentów na mniejszych modelach.

## Konfiguracja LLM

LlamaIndex ma dedykowany wrapper `Ollama` (pakiet `llama-index-llms-ollama`), który uderza prosto w natywne API Ollamy (`POST /api/chat`). Domyślnie próbuje on korzystać z ustrukturyzowanego pola `tools` Ollamy - co dla Bielika nie zadziała (pole `tool_calls` w odpowiedzi pozostaje puste). Wyłączamy więc flagę `is_function_calling_model` - `ReActAgent` automatycznie zejdzie wtedy na ścieżkę parsowania tekstu Thought/Action/Observation.

In [ ]:
from llama_index.llms.ollama import Ollama

bielik_llm = Ollama(
    model="bielik-tools",
    base_url="http://host.docker.internal:11434", # natywne API Ollamy
    is_function_calling_model=False, # wymuszamy ścieżkę ReAct - bez ustrukturyzowanych tool_calls
    temperature=0.3,
    request_timeout=120.0,
    context_window=8192,
)

## Narzędzia i agent

`FunctionTool.from_defaults` zamienia zwykłą funkcję Pythona na narzędzie LlamaIndexa - tak samo jak w DSPy nazwa, sygnatura i docstring funkcji są wszystkim, czego framework potrzebuje. `ReActAgent` zarządza pętlą Thought/Action/Observation pod spodem.

W llama-index-core 0.14+ `ReActAgent` jest workflow-em - tworzymy go bezpośrednio konstruktorem (dawne `ReActAgent.from_tools(...)` zostało usunięte).

In [ ]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.tools import FunctionTool

geolocation_tool = FunctionTool.from_defaults(fn=get_geolocation)
weather_tool = FunctionTool.from_defaults(fn=get_current_weather)

system_prompt = (
    "Jesteś pomocnym asystentem pogodowym. Jeśli do realizacji polecenia "
    "musisz ustalić współrzędne geograficzne jakiegoś miejsca, zawsze "
    "pobierz je za pomocą narzędzia get_geolocation - nigdy nie podawaj "
    "współrzędnych z własnej wiedzy. Nie każde polecenie wymaga "
    "ustalania współrzędnych."
)

agent = ReActAgent(
    tools=[geolocation_tool, weather_tool],
    llm=bielik_llm,
    system_prompt=system_prompt,
    verbose=False,
)

## Uruchomienie agenta

`agent.run(...)` jest wejściem do workflow-owej pętli ReAct - zwraca obiekt typu `WorkflowHandler`, który jest awaitable, więc używamy `await`. Dzięki `verbose=True` zobaczymy każdy krok myślenia agenta i wynik każdego wywołanego narzędzia bezpośrednio na wyjściu.

In [ ]:
# agent.run(...) zwraca workflow handler, który jest awaitable.
# Top-level await działa w Jupyterze.
# Uwaga: IDE może pokazać deprecation hint dla `user_msg=...` - to artefakt
# rozwiązywania overloadów w llama-index 0.14 (sama wiadomość deprecation
# rekomenduje tę samą formę wywołania). Brak ostrzeżenia w runtime.
response = await agent.run(user_msg="Opisz jaka jest pogoda w Poznaniu. Czy powinienem wychodzić na spacer w stroju plażowym i okularach przeciwsłonecznych?")
print("\n=== Ostateczna odpowiedź agenta ===")
print(str(response))

# Wariant 2: agent z function callingiem (`FunctionAgent`)

`FunctionAgent` używa **natywnego function callingu** modelu - oczekuje, że LLM zwróci strukturyzowane pole `tool_calls` w odpowiedzi API, zamiast wkleić wywołanie jako tekst do `content`. Dla Bielika na Ollamie ta ścieżka działa wyłącznie dzięki naszemu customowemu Modelfile (`010-00. Bielik.Modelfile`), który nadpisuje template tak, żeby Ollama umiała wyciągnąć `<tool_call>{...}</tool_call>` z odpowiedzi modelu.

Różnice względem `ReActAgent`:
- mniej tokenów (model nie musi werbalizować `Thought:`/`Action:`/`Observation:`),
- może wywołać kilka narzędzi w jednej turze (równolegle),
- zwykle szybszy i tańszy w produkcji,
- ale działa tylko z modelem+serwerem, które poprawnie zwracają strukturyzowane `tool_calls`.

## Konfiguracja LLM dla `FunctionAgent`

Tworzymy drugą instancję wrappera `Ollama` z `is_function_calling_model=True` - to przełącznik, który mówi LlamaIndexowi: nie schodź na ścieżkę ReActa, używaj strukturyzowanego pola `tools` i czytaj `tool_calls` z odpowiedzi. Bez customowego Modelfile (`010-00. Bielik.Modelfile`) ta flaga była dla Bielika bezużyteczna - serwer i tak zwracał `tool_calls: null` i agent zatrzymywał się od razu, jak omówiono w `010-02`.

In [ ]:
bielik_llm_fc = Ollama(
    model="bielik-tools",
    base_url="http://host.docker.internal:11434",
    is_function_calling_model=True, # włączamy strukturyzowane tool_calls
    temperature=0.3,
    request_timeout=120.0,
    context_window=8192,
)

## Utworzenie i uruchomienie `FunctionAgent`

API jest celowo bardzo podobne do `ReActAgent` - reusujemy te same `geolocation_tool`, `weather_tool` i `system_prompt` zdefiniowane wyżej. Jedyne różnice to klasa agenta i przepięty `llm`.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

function_agent = FunctionAgent(
    tools=[geolocation_tool, weather_tool],
    llm=bielik_llm_fc,
    system_prompt=system_prompt,
    verbose=False,
)

response = await function_agent.run(user_msg="Opisz jaka jest pogoda w Poznaniu. Czy powinienem wychodzić na spacer w stroju plażowym i okularach przeciwsłonecznych?")
print("\n=== Ostateczna odpowiedź agenta ===")
print(str(response))